# Stage 04 — frozen MPNet evaluation companion

## tl;dr

The frozen MPNet trial **did not improve the reference**. Prompt-grouped development balanced accuracy was **14.5946%** for MPNet alone and **34.1153%** when added to character/style features, versus the frozen **35.4131%** reference. Both additions are rejected; the official S01 public score remains **35.4%**.

This companion independently recomputes metrics and intervals from saved out-of-fold predictions. It does not rerun encoder inference or fitting. The full decision is in `results/stage_04/REPORT.md`.

## Context & Methods

The prospective [protocol](../plans/STAGE_04_FROZEN_ENCODER.md) fixes two candidates: normalized frozen MPNet embeddings alone and MPNet added to the character/style model. Each uses balanced LinearSVC, C=1, and the same five prompt-grouped folds. The pretrained encoder is fixed independently of the competition labels; character vocabulary, IDF and style scaling are learned inside each training fold.

### Key Assumptions

The five folds are reused development data. Bootstrap intervals describe the saved predictions conditional on fitted models; they exclude retraining and selection uncertainty. The final private test result is unknown. This companion intentionally shows exact-value tables for three models rather than an additional chart.

## Data

Only saved IDs, prompt groups, labels and held-out predictions are loaded. Raw passage text and model weights are not needed to inspect this evaluation. Run from the repository root or its `notebooks/` directory.

In [1]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from IPython.display import display, HTML

def show_table(frame):
    display(HTML(frame.to_html(border=0, float_format=lambda x: f"{x:.4f}")))

ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p/'results/stage_04/metrics.json').is_file())
OUT = ROOT/'results/stage_04'
metrics = json.loads((OUT/'metrics.json').read_text())
provenance = json.loads((OUT/'encoder_provenance.json').read_text())
assert metrics['status'] == 'complete'
oof = pd.read_csv(OUT/'oof_predictions.csv', dtype={'id':str,'prompt_id':str,'model':str})
prior_path = ROOT/'results/stage_03/oof_predictions.csv'
assert hashlib.sha256(prior_path.read_bytes()).hexdigest() == metrics['previous_predictions_sha256']
prior = pd.read_csv(prior_path, dtype={'id':str,'prompt_id':str,'model':str})
assert oof[['id','prompt_id','model','fold','character_plus_style']].equals(prior[['id','prompt_id','model','fold','character_plus_style']])
assert oof.id.is_unique and len(oof) == 2699
assert oof.prompt_id.nunique() == 150 and oof.groupby('prompt_id').fold.nunique().eq(1).all()
assert set(oof.fold) == {1,2,3,4,5}
labels = sorted(oof.model.unique())
assert len(labels) == 18
print(f"Verified {len(oof):,} held-out predictions per variant; 150 grouped prompts; 18 model labels.")
print('Pinned encoder revision:', provenance['revision'])
print('Truncated passages:', provenance['truncated_rows'])

Verified 2,699 held-out predictions per variant; 150 grouped prompts; 18 model labels.
Pinned encoder revision: e8c3b32edf5434bc2275fc9bab85f82640a19130
Truncated passages: 0


## Results

### Pooled balanced accuracy

Scores are reconciled both directly and as the mean of the 18 class recalls. The first reference row reuses the frozen Stage 03 predictions.

In [2]:
variants = ['character_plus_style','mpnet_only','character_style_mpnet']
recorded = {r['variant']:r['balanced_accuracy'] for r in metrics['comparisons']}
rows = []
for variant in variants:
    score = balanced_accuracy_score(oof.model,oof[variant])
    cm = confusion_matrix(oof.model,oof[variant],labels=labels)
    assert np.isclose(score,(cm.diagonal()/cm.sum(axis=1)).mean(),atol=1e-12)
    assert np.isclose(score,recorded[variant],atol=1e-12)
    rows.append({'Variant':variant,'Balanced accuracy (%)':100*score,
                 'Change from Stage 03 (pp)':100*(score-recorded['character_plus_style'])})
comparison = pd.DataFrame(rows).set_index('Variant')
show_table(comparison.round(4))
print('Fit warnings:',len(metrics['warnings']))
print('Execution time (seconds):',round(metrics['total_seconds'],1))

Fit warnings: 0
Execution time (seconds): 545.1


,Balanced accuracy (%),Change from Stage 03 (pp)
Variant,,
character_plus_style,35.4131,0.0000
mpnet_only,14.5946,-20.8185
character_style_mpnet,34.1153,-1.2978


### Fold consistency and paired uncertainty

Each validation fold holds out 30 complete prompts. Intervals use 2,000 paired resamples of prompts, seed 20261016.

In [3]:
folds = pd.read_csv(OUT/'fold_scores.csv')
assert len(folds) == 15
for _,row in folds.iterrows():
    selected = oof.fold.eq(row['fold'])
    assert int(selected.sum()) == row['rows']
    assert oof.loc[selected,'prompt_id'].nunique() == row['prompts'] == 30
    assert np.isclose(balanced_accuracy_score(oof.loc[selected,'model'],oof.loc[selected,row['variant']]), row['balanced_accuracy'],atol=1e-12)
show_table((100*folds.pivot(index='fold',columns='variant',values='balanced_accuracy')).round(3))

prompts = sorted(oof.prompt_id.unique())
gi = oof.prompt_id.map({x:i for i,x in enumerate(prompts)}).to_numpy()
yi = oof.model.map({x:i for i,x in enumerate(labels)}).to_numpy()
totals = np.zeros((len(prompts),len(labels)),dtype=np.int64)
np.add.at(totals,(gi,yi),1)
weights = np.random.default_rng(20261016).multinomial(len(prompts),np.full(len(prompts),1/len(prompts)),size=2000)
denominator = weights@totals
assert (denominator>0).all()
boot = {}
for variant in variants:
    correct = np.zeros_like(totals)
    np.add.at(correct,(gi,yi),(oof[variant].to_numpy()==oof.model.to_numpy()).astype(int))
    boot[variant] = ((weights@correct)/denominator).mean(axis=1)
intervals = pd.read_csv(OUT/'paired_differences.csv')
for _,row in intervals.iterrows():
    low,high = np.quantile(boot[row.candidate]-boot[row.reference],[.025,.975])
    assert np.allclose([low,high],[row.lower_95,row.upper_95],atol=1e-12)
view = intervals.copy()
for col in ['delta_balanced_accuracy','lower_95','upper_95']:view[col] *= 100
view = view.rename(columns={'delta_balanced_accuracy':'Change (pp)','lower_95':'95% lower (pp)','upper_95':'95% upper (pp)'})
show_table(view.round(4))

variant,character_plus_style,character_style_mpnet,mpnet_only
fold,,,
1,33.5190,33.5190,15.0000
2,33.1480,30.7410,13.7040
3,37.5930,37.0370,14.6300
4,39.2590,35.3700,16.2960
5,33.5500,33.9020,13.3590


,candidate,reference,Change (pp),95% lower (pp),95% upper (pp)
0,mpnet_only,character_plus_style,-20.8185,-22.7179,-19.1111
1,character_style_mpnet,character_plus_style,-1.2978,-2.6324,-0.0370


### Length and class diagnostics

Length groups were fixed before this trial. Scores for each group are full 18-class scores only if all 18 classes are represented. The complete per-model recall table is retained in the results directory.

In [4]:
lengths = pd.read_csv(OUT/'length_diagnostics.csv')
assert lengths.classes_present.eq(18).all()
show_table((100*lengths.pivot(index='word_count_bin',columns='variant',values='balanced_accuracy_present_classes')).reindex(['<=80 words','81-150 words','>150 words']).round(3))
recall = pd.read_csv(OUT/'per_model_recall.csv')
assert len(recall)==54 and recall.support.sum()==3*2699
view = 100*recall.pivot(index='model',columns='variant',values='recall')
view['Change (pp)'] = view['character_style_mpnet']-view['character_plus_style']
show_table(view.sort_values('Change (pp)').round(3))

variant,character_plus_style,character_style_mpnet,mpnet_only
word_count_bin,,,
<=80 words,28.0590,27.0150,10.7420
81-150 words,36.2260,35.5540,16.0400
>150 words,43.4220,40.8450,17.9730


variant,character_plus_style,character_style_mpnet,mpnet_only,Change (pp)
model,,,,
claude-opus-4-6,48.0000,39.3330,10.0000,-8.6670
gpt-5.6-terra,24.0000,19.3330,10.0000,-4.6670
claude-opus-5,60.0000,55.3330,38.0000,-4.6670
claude-sonnet-5,36.6670,32.0000,7.3330,-4.6670
claude-fable-5,15.4360,11.4090,5.3690,-4.0270
gemini-3.1-pro-preview,78.6670,74.6670,26.0000,-4.0000
gemini-3.7-flash,42.6670,40.0000,22.0000,-2.6670
gemini-3.8-flash,40.6670,38.6670,24.0000,-2.0000
gpt-5.6-sol,16.6670,15.3330,6.0000,-1.3330


## Takeaways

Retain Stage 03 as the development reference. The added MPNet block lowered balanced accuracy by 1.2978 percentage points; its paired prompt-bootstrap 95% interval is -2.6324 to -0.0370 points. This is a conditional development comparison, not an independent confirmatory result. All texts fit within the 384-token limit, so truncation did not cause this result. See `results/stage_04/REPORT.md`. A frozen semantic representation need not preserve the model-specific style signals we need. Neither a development gain nor a loss establishes the value of supervised fine-tuning.

### Reproduce the modelling trial

Install `requirements-encoder.txt`, download the pinned public checkpoint once, then run the offline comparison into a **new** output directory:

```bash
python scripts/download_stage04_encoder.py --model-dir models/all-mpnet-base-v2
python scripts/frozen_encoder_comparison.py --data-dir data --model-dir models/all-mpnet-base-v2 --cache-dir encoder-cache --out-dir results/stage_04_reproduction
```

The original trial refuses to overwrite its saved evidence. The notebook itself can be rerun with only the baseline scientific Python dependencies and the saved result files.